# Анализ CRM по сегментам бизнеса

Файл на сервере: `crm_last_version.csv`

**Сегментация по `X_AREA_RESP`:**

| X_AREA_RESP | Сегмент |
|-------------|--------|
| ДМСБ (микро) | МкБ |
| ДМСБ (малый) | МСБ |
| ДМСБ (средний) | МСБ |
| ДКБ | ККБ |

**Период:** 2023-2025 включительно  
**Дедупликация:** уникальные комбинации `X_INN` + `NUMBER_FP_SFP` + `IDENTIFICATION_DATE`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../sources"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


print("Параметры заданы.")

## 1. Загрузка и подготовка данных

In [ ]:
df = pd.read_csv(CRM_FILE, sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print(f"\nНемаппированные значения X_AREA_RESP (будут исключены):")
    print(unmapped.to_string())

df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + NUMBER_FP_SFP + DATE): {len(df):,} строк (удалено {before_dedup - len(df):,} дублей)")

df["year_month"] = df["dt"].dt.to_period("M")

## 2. Проверка: ИНН в нескольких сегментах

In [ ]:
seg_per_inn = df.groupby("inn")["segment"].nunique()
multi_seg = seg_per_inn[seg_per_inn > 1]

print("=" * 60)
print("ПРОВЕРКА: ИНН в нескольких сегментах")
print("=" * 60)
print(f"  Всего уникальных ИНН:              {len(seg_per_inn):,}")
print(f"  ИНН в одном сегменте:              {(seg_per_inn == 1).sum():,}")
print(f"  ИНН в нескольких сегментах:        {len(multi_seg):,}")

if len(multi_seg) > 0:
    print(f"\n  Детали (до 15 примеров):")
    print("  " + "-" * 56)
    for inn in multi_seg.index[:15]:
        subset = df[df["inn"] == inn]
        segs = subset["segment"].unique()
        areas = subset["X_AREA_RESP"].unique()
        n = len(subset)
        print(f"  ИНН {inn}  ->  {n} записей")
        print(f"    X_AREA_RESP: {', '.join(areas)}")
        print(f"    Сегменты:    {', '.join(segs)}")
else:
    print("\n  Каждый ИНН принадлежит ровно одному сегменту.")

## 3. Уникальные ИНН по сегментам

In [ ]:
total_inn = df["inn"].nunique()
seg_order = ["МкБ", "МСБ", "ККБ"]
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(seg_order, fill_value=0)

print("=" * 50)
print("УНИКАЛЬНЫЕ ИНН")
print("=" * 50)
print(f"  Всего:  {total_inn:,}")
print(f"  " + "-" * 30)
for seg in seg_order:
    cnt = inn_by_seg.get(seg, 0)
    pct = cnt / total_inn * 100 if total_inn else 0
    print(f"  {seg:<6s} {cnt:>8,}  ({pct:.1f}%)")

tbl = pd.DataFrame({
    "Сегмент": seg_order,
    "Уникальных ИНН": [inn_by_seg.get(s, 0) for s in seg_order],
})
tbl.loc[len(tbl)] = ["ИТОГО", total_inn]
display(tbl.style.hide(axis="index"))

## 4. Количество ФП/СФП по сегментам

In [ ]:
fp_stats = df.groupby(["segment", "TYPE_FP"]).size().unstack(fill_value=0)
fp_stats = fp_stats.reindex(seg_order, fill_value=0)

for col in ["ФП", "СФП"]:
    if col not in fp_stats.columns:
        fp_stats[col] = 0
fp_stats = fp_stats[["ФП", "СФП"]]
fp_stats["Всего"] = fp_stats["ФП"] + fp_stats["СФП"]

print("=" * 50)
print("КОЛИЧЕСТВО ФП/СФП ПО СЕГМЕНТАМ")
print("=" * 50)

totals = fp_stats.sum()
fp_stats.loc["ИТОГО"] = totals
display(fp_stats)

fp_plot = fp_stats.drop("ИТОГО")
ax = fp_plot[["ФП", "СФП"]].plot(kind="bar", figsize=(8, 4), edgecolor="white")
ax.set_title("Количество ФП и СФП по сегментам", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Количество")
ax.set_xticklabels(seg_order, rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Динамика ФП по месяцам (по сегментам)

In [ ]:
df_fp = df[df["TYPE_FP"] == "ФП"].copy()

pivot_fp = df_fp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
pivot_fp = pivot_fp.reindex(columns=seg_order, fill_value=0)
pivot_fp.index = pivot_fp.index.astype(str)

print("=" * 60)
print("ДИНАМИКА ФП ПО МЕСЯЦАМ")
print("=" * 60)
display(pivot_fp)

fig, ax = plt.subplots(figsize=(14, 5))
pivot_fp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
ax.set_title("Количество выявленных ФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество ФП")
ax.legend(title="Сегмент")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Динамика СФП по месяцам (по сегментам)

In [ ]:
df_sfp = df[df["TYPE_FP"] == "СФП"].copy()

pivot_sfp = df_sfp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
pivot_sfp = pivot_sfp.reindex(columns=seg_order, fill_value=0)
pivot_sfp.index = pivot_sfp.index.astype(str)

print("=" * 60)
print("ДИНАМИКА СФП ПО МЕСЯЦАМ")
print("=" * 60)
display(pivot_sfp)

fig, ax = plt.subplots(figsize=(14, 5))
pivot_sfp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
ax.set_title("Количество выявленных СФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество СФП")
ax.legend(title="Сегмент")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()